In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from baby_pytorch import Tensor

np.random.seed(42)

In [ ]:
def get_labels(input_data, radius):
    squared_distance = np.sum(input_data ** 2, axis=1, keepdims=True)
    return np.where(squared_distance < radius ** 2, -1.0, 1.0)


num_samples = 5000
radius = 0.5
inputs = Tensor(np.random.uniform(-1, 1, size=(num_samples, 2)))
labels = Tensor(get_labels(inputs.data, radius))

In [ ]:
def plot_data(inputs, labels):
    plt.scatter(
        inputs.data[:, 0],
        inputs.data[:, 1],
        c=labels.data.ravel(),
        cmap="bwr",
    )
    plt.gca().set_aspect("equal")
    plt.show()

In [ ]:
plot_data(inputs, labels)

In [ ]:
from baby_pytorch.loss import MSE
from baby_pytorch.nn import MLP, Tanh
from baby_pytorch.optim import SGD

num_epochs = 20
batch_size = 50

mlp = MLP(2, [16, 8], 1, Tanh())
optimizer = SGD(mlp.parameters(), lr=0.1)
loss_history = []

for epoch in range(num_epochs):
    epoch_loss = 0.0
    for idx in range(0, num_samples, batch_size):
        batch_inputs = inputs[idx:idx + batch_size]
        batch_labels = labels[idx:idx + batch_size]

        optimizer.zero_grad()
        predictions = mlp(batch_inputs)
        loss = MSE(predictions, batch_labels)
        loss.backward()
        optimizer.step()

        epoch_loss += float(loss.data) * batch_inputs.shape[0]

    epoch_loss /= num_samples
    loss_history.append(epoch_loss)
    print(f"Epoch {epoch + 1:2d}: loss={epoch_loss:.6f}")

In [ ]:
def get_predictions(logits):
    return np.where(logits.data < 0, -1.0, 1.0)


def calculate_accuracy(predictions, labels):
    return 100.0 * np.mean(predictions == labels.data)

In [ ]:
train_logits = mlp(inputs, training=False)
train_predictions = get_predictions(train_logits)
train_accuracy = calculate_accuracy(train_predictions, labels)
print(f"Train accuracy: {train_accuracy:.2f}%")

In [ ]:
num_test_samples = 1000
test_inputs = Tensor(np.random.uniform(-1, 1, size=(num_test_samples, 2)))
test_labels = Tensor(get_labels(test_inputs.data, radius))
plot_data(test_inputs, test_labels)

In [ ]:
def plot_errors(inputs, labels, predictions):
    wrong = predictions.ravel() != labels.data.ravel()
    plt.scatter(
        inputs.data[:, 0],
        inputs.data[:, 1],
        c=labels.data.ravel(),
        cmap="bwr",
        edgecolors="k",
        s=30,
    )
    plt.scatter(
        inputs.data[wrong, 0],
        inputs.data[wrong, 1],
        c="lime",
        edgecolors="k",
        s=60,
        zorder=3,
    )
    plt.gca().set_aspect("equal")
    plt.show()

In [ ]:
test_logits = mlp(test_inputs, training=False)
test_predictions = get_predictions(test_logits)
test_accuracy = calculate_accuracy(test_predictions, test_labels)
print(f"Test accuracy: {test_accuracy:.2f}%")
plot_errors(test_inputs, test_labels, test_predictions)

In [ ]:
grid_axis = np.linspace(-1, 1, 50)
grid_x, grid_y = np.meshgrid(grid_axis, grid_axis)
grid_inputs = Tensor(np.column_stack((grid_x.ravel(), grid_y.ravel())))
grid_logits = mlp(grid_inputs, training=False)
grid_values = grid_logits.data.reshape(grid_x.shape)

plt.contourf(
    grid_x,
    grid_y,
    grid_values,
    levels=[-1e9, 0, 1e9],
    colors=["#cfd8ff", "#ffd0d0"],
)
plt.scatter(
    inputs.data[:, 0],
    inputs.data[:, 1],
    c=labels.data.ravel(),
    cmap="bwr",
    edgecolors="k",
    s=30,
)
plt.gca().set_aspect("equal")

theta = np.linspace(0, 2 * np.pi, 200)
plt.plot(
    radius * np.cos(theta),
    radius * np.sin(theta),
    "k--",
)
plt.show()